# PL DMA through the HP port
The PL copies buffers in DDR (and sums them) over the PS high-performance port. Buffers come from the `zmbuf` driver (physically contiguous, uncached), as numpy arrays. Addresses and lengths must be multiples of 128 bytes; misaligned requests are rejected.

In [ ]:
import numpy as np, time
from zynqmini import Overlay, HpDma, allocate
ol = Overlay('zm_base')
dma = HpDma(ol)
n = 1 << 20
a = allocate((n // 4,), 'uint32'); b = allocate((n // 4,), 'uint32')
a.array[:] = np.random.default_rng(1).integers(0, 2**32, n // 4, dtype=np.uint32)
b.array[:] = 0
a.phys_addr, b.phys_addr

In [ ]:
t = time.perf_counter(); dma.copy(a, b); dt = time.perf_counter() - t
print('copied', n, 'bytes in', round(dt * 1e3, 2), 'ms (includes Python overhead);', 'PL cycles:', ol.DMA_CYCLES, '=', round(n / (ol.DMA_CYCLES / 100e6) / 1e6), 'MB/s at 100 MHz')
print('identical:', np.array_equal(a.array, b.array))

In [ ]:
want = int(a.array.view(np.uint32).sum(dtype=np.uint64)) & 0xFFFFFFFF
print('PL checksum', hex(dma.checksum(a)), 'numpy', hex(want))

## Completion by interrupt
The PL interrupt wakes a sleeping process instead of polling.

In [ ]:
irq = ol.irq()
dma.copy(a, b, irq=irq)
irq.close()
print('done by interrupt, status', ol.DMA_STATUS)

In [ ]:
a.free(); b.free()